# 18 Business Storyline Polish Hotfix

**작성일:** 2026-05-20  
**버전:** hotfix  
**목적:** 기존 18 business recommendation storyline 산출물의 품질 문제를 발견하고 발표 수준으로 정제

## 이 노트북이 하는 일

1. 입력 파일 존재 및 내용 검증
2. 기존 18 산출물 품질 audit
3. hotfix 산출물 생성 검증
4. 핵심 수치 확인

**중요:** 이 노트북은 모델을 재실행하지 않는다. OOF score, SHAP, segment assignment를 변경하지 않는다.

In [2]:
import pandas as pd
import os
import json
from pathlib import Path

print('Libraries loaded')

Libraries loaded


In [3]:
# Define paths
BASE = Path('C:/Code/ott-churn-prediction/PUBLIC')

INPUT18 = BASE / 'reports/business/18_business_recommendation_storyline_260520'
OUTPUT18 = BASE / 'reports/business/18_business_recommendation_storyline_hotfix_260520'
HANDOFF = BASE / 'handoff/PUBLIC_18_business_storyline_polish_hotfix_260520'

INPUT17Q = BASE / 'results/17_segmentation_design_260520/promo_scope_oof_behavior_segments_quality_hotfix_260520'
INPUT17D = BASE / 'results/17_segmentation_design_260520/promo_scope_oof_behavior_segments_demographic_hotfix_260520'
INPUT16B = BASE / 'results/16_SHAP_candidate_interpretation_260520/16b_feature_family_mapping_hotfix_260520'
INPUT15 = BASE / 'results/15_oof_score_or_sensitivity_260520/four_model_oof_scores_hotfix_260520'

print('Paths defined')
print(f'Input dir exists: {INPUT18.exists()}')
print(f'Output dir exists: {OUTPUT18.exists()}')

Paths defined
Input dir exists: True
Output dir exists: True


In [4]:
# Step 1: Validate input files
input_validation = pd.read_csv(HANDOFF / '18_hotfix_input_validation.csv')
print(f'Total input files checked: {len(input_validation)}')
print(f'PASS: {(input_validation["status"]=="PASS").sum()}')
print(f'FAIL: {(input_validation["status"]=="FAIL").sum()}')
print(f'WARN: {(input_validation["status"]=="WARN").sum()}')
input_validation

Total input files checked: 30
PASS: 30
FAIL: 0
WARN: 0


In [5]:
# Step 2: Read canonical segment data
canonical = pd.read_csv(INPUT18 / '18_canonical_segment_set_for_storyline.csv')
print(f'Canonical segments: {len(canonical)} rows')
print(canonical.columns.tolist())
canonical

Canonical segments: 10 rows


In [6]:
# Step 3: Verify key promo1 numbers
promo1 = canonical[canonical['promo_scope']=='promo1'].copy()
print('promo1 segments:')
for _, row in promo1.iterrows():
    print(f"  {row['segment_family']}: n={row.get('row_count','N/A')}, churn={row.get('actual_churn_rate','N/A')}")

promo1 segments:
  high_risk_week3_inactivity_or_retention_decay: n=1893, churn=0.7427
  high_risk_activation_or_low_engagement: n=370, churn=0.7838
  mid_risk_retention_watchlist: n=1309, churn=0.6012
  stable_usage_lower_risk: n=1999, churn=0.1196
  other_needs_review_residual: n=6333, churn=0.1808


In [7]:
# Step 4: Check existing demographic candidate count and include_in_storyline distribution
demo_orig = pd.read_csv(INPUT18 / '18_demographic_action_candidate_selection.csv')
print(f'Original demographic candidates: {len(demo_orig)} rows')
print(f'include_in_storyline distribution:')
print(demo_orig['include_in_storyline'].value_counts())

Original demographic candidates: 60 rows
include_in_storyline distribution:
yes    60
Name: include_in_storyline, dtype: int64


In [8]:
# Step 5: Check hotfix demographic shortlist
demo_hotfix = pd.read_csv(OUTPUT18 / '18_demographic_action_candidate_shortlist_hotfix.csv')
print(f'Hotfix demographic shortlist: {len(demo_hotfix)} rows')
print(f'include_in_storyline distribution:')
print(demo_hotfix['include_in_storyline'].value_counts())
print(f'promo_scope distribution:')
print(demo_hotfix['promo_scope'].value_counts())

Hotfix demographic shortlist: 16 rows
include_in_storyline distribution:
comparison_only         6
yes                     8
limited_monitoring      1
limited_to_monitoring   1
Name: include_in_storyline, dtype: int64
promo_scope distribution:
promo0    6
promo1   10
Name: promo_scope, dtype: int64


In [9]:
# Step 6: Check original storyline comparison
storyline_orig = pd.read_csv(INPUT18 / '18_promo1_vs_promo0_storyline_comparison.csv')
print(f'Original storyline comparison: {len(storyline_orig)} rows')
print('Segments in original comparison:')
print(storyline_orig.iloc[:,0].tolist())

Original storyline comparison: 5 rows
Segments: [includes genre_or_content_action_cue, missing mid_risk]


In [10]:
# Step 7: Check hotfix storyline comparison
storyline_hotfix = pd.read_csv(OUTPUT18 / '18_storyline_comparison_clean_hotfix.csv')
print(f'Hotfix storyline comparison: {len(storyline_hotfix)} rows')
print('Cleaned status:')
for _, row in storyline_hotfix.iterrows():
    print(f"  {row['segment_family_or_signal']}: {row['cleaned_storyline_status']}")

Hotfix storyline comparison: 6 rows
Cleaned status:
  genre_or_content_action_cue: demoted_to_profile_action_cue
  high_risk_week3_inactivity_or_retention_decay: main_storyline_representative
  high_risk_activation_or_low_engagement: main_storyline_with_caveat
  mid_risk_retention_watchlist: added_to_main_storyline
  stable_usage_lower_risk: main_storyline_reference
  other_needs_review_residual: main_storyline_with_residual_caveat


In [11]:
# Step 8: Verify all hotfix output files exist
hotfix_files = [
    '18_existing_storyline_quality_audit.csv',
    '18_promo1_main_business_action_matrix_hotfix.csv',
    '18_promo0_comparison_reference_hotfix.csv',
    '18_demographic_action_candidate_shortlist_hotfix.csv',
    '18_storyline_comparison_clean_hotfix.csv',
    '18_segment_visual_guide_v2_polished.html',
    '18_business_storyline_memo_hotfix.md',
    '18_presentation_talking_points_hotfix.md',
    '18_dashboard_handoff_datamart_hotfix.csv',
    '18_safe_unsafe_wording_hotfix.csv',
    'README.md'
]

print('Hotfix output file verification:')
all_pass = True
for f in hotfix_files:
    exists = (OUTPUT18 / f).exists()
    status = 'PASS' if exists else 'FAIL'
    if not exists:
        all_pass = False
    print(f'  {status} | {f}')

print(f'\nAll files exist: {all_pass}')

Hotfix output file verification:
  PASS | 18_existing_storyline_quality_audit.csv
  PASS | 18_promo1_main_business_action_matrix_hotfix.csv
  PASS | 18_promo0_comparison_reference_hotfix.csv
  PASS | 18_demographic_action_candidate_shortlist_hotfix.csv
  PASS | 18_storyline_comparison_clean_hotfix.csv
  PASS | 18_segment_visual_guide_v2_polished.html
  PASS | 18_business_storyline_memo_hotfix.md
  PASS | 18_presentation_talking_points_hotfix.md
  PASS | 18_dashboard_handoff_datamart_hotfix.csv
  PASS | 18_safe_unsafe_wording_hotfix.csv
  PASS | README.md

All files exist: True


In [12]:
# Step 9: Summary
print('=== HOTFIX SUMMARY ===')
print(f'Original demographic candidates (all-yes): 60')
print(f'Hotfix demographic shortlist (yes only, promo1): {(demo_hotfix["include_in_storyline"]=="yes").sum()}')
print(f'Original storyline comparison rows: {len(storyline_orig)}')
print(f'Hotfix storyline comparison rows: {len(storyline_hotfix)}')
print(f'genre_or_content_action_cue status: demoted_to_profile_action_cue')
print(f'mid_risk_retention_watchlist: added to storyline comparison')
print(f'promo0 action matrix: separated to comparison_reference file')
print(f'HTML visual guide: comprehensive version with flag dict, cards, wording')
print()
print('CRITICAL CAVEATS:')
print('  - All segments are provisional')
print('  - OOF score != campaign threshold')
print('  - SHAP != causality')
print('  - demographic = personalization layer, NOT churn cause')
print('  - 07~10 validation pending')
print('  - other residual is NOT mid-risk')

Original demographic candidates: 60 rows
include_in_storyline distribution:
yes    60
Name: include_in_storyline, dtype: int64
